# 02 - Medical Dictionary Creation

In this notebook, we construct the **Semantic Concept Matrix ($T$)** using a robust, data-driven, and clinically validated pipeline. This matrix serves as the semantic ground truth required for the unsupervised concept discovery and valuation of the Sparse Autoencoder (SAE) latent features.

In [1]:
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

    requirements_path = "/content/drive/MyDrive/xai-project5/requirements.txt"
else:
    requirements_path = "./../../requirements.txt"

!pip install -r "{requirements_path}"

In [2]:
import os

if IN_COLAB:
    from google.colab import userdata

    # Reading secrets from Google Colab
    UMLS_API_KEY = userdata.get('UMLS_API_KEY')
    HF_TOKEN = userdata.get('HF_TOKEN')

    save_dir = '/content/drive/MyDrive/xai-project5/results/02_dictionary_creation'
else:
    from dotenv import load_dotenv

    # Load from .env file in the project folder
    load_dotenv()
    UMLS_API_KEY = os.getenv('UMLS_API_KEY')
    HF_TOKEN = os.getenv('HF_TOKEN')
    
    save_dir = os.path.abspath(os.path.join('..', 'results', '02_dictionary_creation'))

os.makedirs(save_dir, exist_ok=True)
print(f"Save directory configured at: {save_dir}")

Save directory configured at: /Users/riccardo/Library/CloudStorage/GoogleDrive-riccardomarconi01@gmail.com/Il mio Drive/Progetti/04 - Explainable and Trustworthy AI/Project/xai-project5/src/results/02_dictionary_creation


In [4]:
import re
import pandas as pd
from collections import Counter
from datasets import load_dataset

## 1. Candidate Concept Extraction from Open-I

We derive the initial vocabulary directly from the `Problems` metadata of the **Open-I chest X-ray dataset**. Each record may contain multiple indexed terms; we split, normalize, deduplicate, and count these terms to obtain dataset-specific candidate concepts.

This stage produces a broad inventory that can include findings, anatomical structures, devices, descriptive terms, and non-clinical metadata. It is therefore an initial candidate pool—not yet a validated clinical dictionary.

In [5]:
DATASET_ID = "ykumards/open-i"
EXCLUDED_ENTRIES = {"", "normal", "no indexing"}

# Load the Open-I split and access only its textual metadata. Images are not used in this dictionary-extraction step.
print(f"Loading '{DATASET_ID}' metadata...")
dataset_meta = load_dataset(DATASET_ID, split="train")

Loading 'ykumards/open-i' metadata...


In [9]:
def normalize_term(term):
    """Normalize formatting without changing the clinical meaning."""

    term = str(term).strip().lower()
    term = re.sub(r"\s+", " ", term)
    return term

concept_counts = Counter()
raw_examples = {}

for problems in dataset_meta["Problems"]:
    if problems is None:
        continue

    # In Open-I, semicolons separate indexed concepts.
    # We don't split on commas: e.g. "Aorta, Thoracic" is one concept
    for raw_term in str(problems).split(";"):
        concept = normalize_term(raw_term)

        if concept in EXCLUDED_ENTRIES:
            continue

        concept_counts[concept] += 1
        raw_examples.setdefault(concept, raw_term.strip())

candidate_concepts = (
    pd.DataFrame(
        [
            {
                "candidate_term": concept,
                "frequency": count,
                "example_raw_term": raw_examples[concept],
            }
            for concept, count in concept_counts.items()
        ]
    )
    .sort_values(["frequency", "candidate_term"], ascending=[False, True])
    .reset_index(drop=True)
)

# This remains the complete, dataset-derived candidate pool.
dynamic_labels = candidate_concepts["candidate_term"].tolist()

print(f"\nUnique candidate concepts extracted: {len(dynamic_labels)}")
print(f"Total non-empty concept occurrences: {sum(concept_counts.values())}")

print("\nTop 50 candidates by frequency:")
display(candidate_concepts.head(50))


Unique candidate concepts extracted: 116
Total non-empty concept occurrences: 6475

Top 50 candidates by frequency:


,candidate_term,frequency,example_raw_term
0,lung,553,Lung
1,opacity,509,Opacity
2,cardiomegaly,345,Cardiomegaly
3,calcinosis,332,Calcinosis
4,pulmonary atelectasis,330,Pulmonary Atelectasis
5,calcified granuloma,276,Calcified Granuloma
6,thoracic vertebrae,257,Thoracic Vertebrae
7,cicatrix,196,Cicatrix
8,spine,174,Spine
9,markings,167,Markings


In [10]:
# Optional: preserve this intermediate audit artifact.
candidate_concepts.to_csv(
    os.path.join(save_dir, "openi_concepts_raw.csv"),
    index=False,
)

## 2. Clinical Filtration & Domain Alignment

The raw candidate concepts extracted from the Open-I `Problems` field are highly noisy. They contain anatomical structures and pathological entities, but also medical devices, surgical materials, and technical image-quality annotations.

We must rigorously filter this vocabulary, because in the final evaluation we will use a large language model (MedGemma) to evaluate semantic alignment exclusively against the clinical `findings` and `impression` sections of radiology reports, evaluating non-clinical concepts (e.g., "Surgical Instruments") would artificially inflate our `Unaligned` and `Uncertain` metrics.

### Pipeline Actions:
1. **The Exclusion Filter:** We systematically drop hardware, procedures, and image quality descriptors.
2. **Domain Alignment:** Ambiguous terms inherent to the chest X-ray domain (e.g., "Mass", "Opacity") are too broad for a universal ontology like UMLS. We programmatically map these to their precise thoracic/pulmonary counterparts (e.g., "Pulmonary Mass", "Lung Opacity") *before* querying the UMLS API. This guarantees deterministic CUI retrieval and eliminates the need for brittle error-handling loops.

In [11]:
# Load the raw candidate concepts
print("Loading raw concepts from Step 1...")
raw_csv_path = os.path.join(save_dir, "openi_concepts_raw.csv")
candidate_concepts = pd.read_csv(raw_csv_path)

Loading raw concepts from Step 1...


In [12]:
# Devices, surgical hardware, procedures, and technical/image-quality artifacts. Not clinical anatomical/pathological findings.
EXCLUSION_FILTER = {
    # Devices / hardware
    "catheters, indwelling",
    "surgical instruments",
    "implanted medical device",
    "medical device",
    "tube, inserted",
    "stents",
    "sutures",
    "breast implants",
    "foreign bodies",

    # Procedures / post-surgical states
    "mastectomy",
    "pneumonectomy",
    "spinal fusion",
    "colonic interposition",

    # Imaging agents / technical artifacts
    "contrast media",
    "technical quality of image unsatisfactory",
}

# This contextualizes ambiguous Open-I shorthand to strict pulmonary/thoracic terminology
DOMAIN_ALIGNMENT_MAP = {
    # ---- Ambiguous Pathological Descriptors ----
    "opacity":          "lung opacity",
    "density":          "lung density",
    "markings":         "bronchovascular markings",
    "infiltrate":       "pulmonary infiltrate",
    "consolidation":    "pulmonary consolidation",
    "airspace disease": "pulmonary airspace disease",
    "thickening":       "pleural thickening",
    "lucency":          "pulmonary lucency",
    "volume loss":      "lung volume loss",
    "cavitation":       "pulmonary cavitation",
    "fibrosis":         "pulmonary fibrosis",
    "cicatrix":         "pulmonary cicatrix",
    "cysts":            "pulmonary cyst",

    # ---- Focal Lesions ----
    "mass":             "pulmonary mass",
    "nodule":           "pulmonary nodule",
    "granuloma":        "pulmonary granuloma",
    "blister":          "pulmonary bleb",

    # ---- Anatomical Positioning & Structural ----
    "shift":            "mediastinal shift",
    "sulcus":           "costophrenic sulcus",
    "cardiac shadow":   "cardiac silhouette",
    "deformity":        "thoracic skeletal deformity",
    "sclerosis":        "bone sclerosis", 
    "fractures, bone":  "bone fracture",
    "blood vessels":    "pulmonary blood vessels"
}

print("Applying Clinical Filtration and Domain Alignment...")

# Initialize the clean list for Step 3
filtered_rows = []
dropped_terms = []

for _, row in candidate_concepts.iterrows():
    raw_term = row['candidate_term']
    
    # 1. Apply Exclusion Filter
    if raw_term in EXCLUSION_FILTER:
        dropped_terms.append(raw_term)
        continue
        
    # 2. Apply Domain Alignment
    # If the term is in our map, use the specific context; otherwise, keep the original
    aligned_term = DOMAIN_ALIGNMENT_MAP.get(raw_term, raw_term)
    
    filtered_rows.append({
        "aligned_term": aligned_term,
        "frequency": row['frequency'],
        "example_raw_term": row['example_raw_term']
    })

# Convert to DataFrame
clinical_concepts_df = pd.DataFrame(filtered_rows)

# Group by the aligned terms to merge duplicates and sum their frequencies
final_df = clinical_concepts_df.groupby('aligned_term').agg({
    'frequency': 'sum',
    'example_raw_term': 'first'  # Keep the first raw term as an example
}).reset_index().sort_values(by='frequency', ascending=False)

print(f"Original concepts extracted: {len(candidate_concepts)}")
print(f"Concepts dropped (Hardware/Artifacts): {len(dropped_terms)}")
print(f"Final validated clinical concepts ready for UMLS: {len(final_df)}")

print("\nTop 10 validated clinical concepts:")
display(final_df.head(10))

Applying Clinical Filtration and Domain Alignment...
Original concepts extracted: 116
Concepts dropped (Hardware/Artifacts): 15
Final validated clinical concepts ready for UMLS: 100

Top 10 validated clinical concepts:


,aligned_term,frequency,example_raw_term
48,lung,553,Lung
51,lung opacity,509,Opacity
19,cardiomegaly,345,Cardiomegaly
17,calcinosis,332,Calcinosis
70,pulmonary atelectasis,330,Pulmonary Atelectasis
16,calcified granuloma,276,Calcified Granuloma
95,thoracic vertebrae,257,Thoracic Vertebrae
74,pulmonary cicatrix,196,Cicatrix
91,spine,174,Spine
14,bronchovascular markings,167,Markings


In [13]:
# Save the finalized, clean dictionary to a new CSV
filtered_csv_path = os.path.join(save_dir, "openi_concepts_filtered_aligned.csv")
final_df.to_csv(filtered_csv_path, index=False)
print(f"\nFiltered clinical concepts successfully saved to: {filtered_csv_path}")


Filtered clinical concepts successfully saved to: /Users/riccardo/Library/CloudStorage/GoogleDrive-riccardomarconi01@gmail.com/Il mio Drive/Progetti/04 - Explainable and Trustworthy AI/Project/xai-project5/src/results/02_dictionary_creation/openi_concepts_filtered_aligned.csv
